# SRM / Trade Promotion Optimisation: Walkthrough

A step-by-step run of the demand-modelling and promo-optimisation pipeline.
Everything uses **synthetic** scanner data with *known* elasticities, so we can check
that the model recovers them.

**Unit of analysis:** CTA (market) × PPG (product price group) × week.


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../src'))
import pandas as pd, config as C
pd.set_option('display.max_columns', 40)


## 1. Generate sample data
Realistic scanner data with a log-log data-generating process and baked-in true elasticities.


In [ ]:
import s00_generate_sample_data as s00
raw = s00.generate()
raw.head()


## 2. Union & map
Append extracts, run raw-data QC, map dimension keys and financials.


In [ ]:
import s01_union_map as s01
ads = s01.run()
ads[['cta','ppg','week_ending','units','avg_price','base_price','list_price','cogs_per_unit']].head()


## 3. Treatment & feature engineering
Build log-log features: `log_price`, feature/display %ACV, seasonality, trend, pantry lag,
competitor inter/intra log-prices, and a time-based train/test flag.


In [ ]:
import s02_treatment as s02
m = s02.run()
[c for c in m.columns if c not in ('manufacturer','brand','category','segment')]


## 4. Log-log stepwise model (per CTA-PPG)
`log(units) ~ log_price + feature + display + seasonality + trend + pantry + competitors`,
with stepwise (AIC), VIF and economic **sign constraints**. The `price_elasticity` column is
the coefficient on `log_price`.


In [ ]:
import s03_modelling as s03
summary = s03.run()
summary.sort_values('price_elasticity')[['cta_ppg','r2','test_mape','price_elasticity','price_sign_valid']].head(10)


### Did it work? Recovered vs TRUE elasticity
Because we generated the data, we know the right answer.


In [ ]:
truth = pd.read_csv(C.DATA/'ground_truth_elasticities.csv')
rec = summary.groupby('ppg').price_elasticity.mean().reset_index()
chk = rec.merge(truth, on='ppg'); chk['abs_err'] = (chk.price_elasticity - chk.true_beta_price).abs()
chk.round(3).sort_values('ppg')


## 5. Baseline vs incremental decomposition
Predict with the promo levers zeroed → baseline; actual − baseline = incremental lift.


In [ ]:
import s04_baseline as s04
decomp = s04.run()
decomp.head()


## 6. Insights: waterfall, promo efficiency, price ladder


In [ ]:
import s05_insights as s05
eff = s05.run()
eff[eff.manufacturer==C.FOCAL_MANUFACTURER][['ppg','promo_weeks','lift_pct','roi']]


## 7. Optimiser: best promo calendar under guardrails
Reallocate the promo budget to the highest-return weeks/depths subject to margin, trade-spend,
AUP and no-promo-week guardrails (the OR-Tools version is in `databricks/06_OPTIMISER.py`).


In [ ]:
import s06_optimiser as s06
reco = s06.run()
reco[['ppg','cta','current_profit','optimised_profit','uplift_pct']].head(10)


## 8. Quality checks & validation


In [ ]:
import s07_quality_checks as s07
qc = s07.run()
qc


## 9. Charts & dashboard


In [ ]:
import viz, report
viz.build_all(); report.build_dashboard()
print('Open outputs/dashboard.html')


---
**Takeaway:** the pipeline recovers known elasticities (mean abs error ≈ 0.03), decomposes
sales into baseline vs promo lift, ranks promotions by ROI, and the optimiser lifts the modelled
profit pool while cutting trade spend.
